# Using RNNs With Keras on Mathematical Equations

## Checking GPU Usage

Go to the `Runtime > Change runtime type` menu and check that we're on Python 3 and that the hardware accelerator is set to "GPU".

In [ ]:
!nvidia-smi

## Importing TensorFlow and the Other Required Libraries

In [ ]:
import operator
import random
import typing

import matplotlib.pyplot as plt
import numpy
import seaborn
import sklearn.model_selection
import keras

## Defining the Vocabulary

In this lab, we'll define equations such as `3 + 1 = 4`. The model will have to predict `4` given `3 + 1` as input.

To start, let's create the vocabulary:

In [ ]:
operations = list("+*-/")
numbers = list("0123456789.")
padding = [" "]

index_to_char = numbers + operations + padding
char_to_index = {c: i for i, c in enumerate(index_to_char)}

print(f"Index to character: {index_to_char}")
print(f"Character to index: {char_to_index}")

## Using the Vocabulary to Encode and Decode Equations

We can now use this vocabulary to turn text equations into sequences of numbers that a neural network can understand:

In [ ]:
def encode(characters: str) -> numpy.ndarray:
  return numpy.array([char_to_index[char] for char in characters])


def decode(array: numpy.ndarray):
  return ''.join(index_to_char[i] for i in array)

*Test the `encode` and `decode` functions. Do you think the neural network will be able to work directly on the output of `encode`, or will additional preprocessing be needed?*

In [ ]:
# Your test code here

Your answer here

### Solution

In [ ]:
equation = "3/4+2-5"
result = "0"
print("─" * 50)
print("Equation")
print("─" * 50)
print(f"Raw form          {equation}")
print(f"Encoding          {encode(equation)}")
print(f"Encoding/decoding {decode(encode(equation))}")
print()
print("─" * 50)
print("Result")
print("─" * 50)
print(f"Raw form          {result}")
print(f"Encoding          {encode(result)}")
print(f"Encoding/decoding {decode(encode(result))}")

This encoding isn't enough for neural network processing: you'll need to either one-hot encode on top of it, or use an embedding layer.

## Generating Data

We can now generate examples, which will be used for training, validation and testing.

To do so, we'll randomly pick one of the 4 defined operations and generate random integers to apply it to. This process is repeated until we reach the desired number of examples.

In [ ]:
def make_maths_problem(n_samples=1000,
                       n_digits=3,
                       invert=True):
  equations = []
  results = []

  math_operation = {"+": operator.add,
                    "-": operator.sub,
                    "*": operator.mul,
                    "/": operator.truediv}

  # Maximum length of the string describing the operation
  max_equation_len = 2 * n_digits + 1
  max_result_len = 2 * n_digits

  while len(equations) < n_samples:
    # Pick a random operation
    operation = numpy.random.choice(operations)

    # Generate two integers within the size limit
    left, right = numpy.random.randint(10 ** n_digits, size=2)

    equation = f"{left}{operation}{right}"

    if equation not in equations:
      # Compute the result. Since left and right are numpy integers,
      # this computation raises no exception, even on division by 0
      math_result = math_operation[operation](left, right)

      # Start over if the result isn't usable
      if math_result == numpy.inf or numpy.isnan(math_result):
        continue

      # The result can be very long (0.3333333…): limit its length
      result = str(math_result)[:max_result_len]

      # "Pad" so that all sequences have the same length
      padded_equation = equation.ljust(max_equation_len)
      padded_result = result.ljust(max_result_len)

      # Invert if the invert argument is given
      if invert:
        padded_equation = padded_equation[::-1]

      equations.append(padded_equation)
      results.append(padded_result)

  X = numpy.array(list(map(encode, equations)))
  y = numpy.array(list(map(encode, results)))
  return X, y

Let's test this function with a dozen examples:

In [ ]:
X, y = make_maths_problem(10, n_digits=3)

print(f"X shape:  {X.shape}")
print(f"y shape:  {y.shape}")

print()
print("A few generated examples")
for encoded_equation, encoded_result in zip(X, y):
    # Equation inversion is enabled by default: undo it
    # to display the original equation
    equation = decode(encoded_equation[::-1])
    result = decode(encoded_result)
    print(f"{equation} = {result}")

We can now create the dataset and the necessary splits:

In [ ]:
X, y = make_maths_problem(100_000, n_digits=3)
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
    X, y, train_size=0.8)

## Defining the Model

To process these character sequences, we'll use an RNN. As said before, the input sequences will need to be one-hot encoded or passed through an embedding layer. Here, we'll use the embedding layer.

In [ ]:
def rnn_model(hidden_size=1024, n_digits=3):
  model = keras.models.Sequential()
  # Encoder
  model.add(keras.layers.Embedding(len(index_to_char), 16))
  model.add(keras.layers.GRU(hidden_size, return_sequences=True))
  model.add(keras.layers.GRU(hidden_size))

  # Decoder
  # The RepeatVector layer duplicates the encoder's output as many times
  # as we want output characters. It's a simple way to
  # condition the decoding of the result on the encoding of the equation
  model.add(keras.layers.RepeatVector(2 * n_digits))
  # Note the return_sequences=True argument: without it, we would
  # only produce a single output
  model.add(keras.layers.GRU(hidden_size, return_sequences=True))

  # Apply a dense output layer at each timestep
  model.add(keras.layers.TimeDistributed(
      keras.layers.Dense(len(index_to_char), activation="softmax")))

  model.compile(loss="sparse_categorical_crossentropy",
                optimizer="adam",
                metrics=["accuracy"])
  return model


model = rnn_model()

## Training the Model

We'll regularly display predictions on fixed examples to see how the model evolves.

In [ ]:
# We'll use the same examples at each evaluation to see the model
# progress
n_test = 20
indexes = numpy.random.choice(X_test.shape[0], replace=False, size=n_test)
X_examples, y_examples = X_test[indexes], y_test[indexes]

In [ ]:
def examples() -> None:
  guesses = numpy.argmax(model.predict(X_examples, verbose=0), axis=-1)
  for encoded_equation, encoded_result, encoded_guess in zip(
      X_examples, y_examples, guesses):
    equation = decode(encoded_equation[::-1])
    result = decode(encoded_result)
    guess = decode(encoded_guess)
    print(f"{equation} = {result} ?= {guess}")

for i in range(10):
  model.fit(X_train,
            y_train,
            batch_size=4096,
            epochs=10,
            verbose=True,
            validation_data=(X_test, y_test))
  print()
  print("─" * 50)
  print(f"After {(i + 1) * 10} epochs:")
  examples()

## Evaluating the Model

To evaluate our model, we'll compute 3 things on the test set:

- the illegal predictions (when the model's output can't be interpreted as a float)
- the accuracy
- the fractional error ($\frac{y_{pred} - y}{y}$)

In [ ]:
def evaluate() -> None:
  # Compute the model's predictions
  predictions = numpy.argmax(model.predict(X_test), axis=-1)

  # Decode the model's predictions into strings
  str_predictions = numpy.apply_along_axis(decode, axis=1, arr=predictions)

  # Decode the model's predictions into floats
  # Sometimes the emitted string doesn't represent a valid float
  # Create a boolean mask to filter these elements afterwards
  illegal_predictions_mask = numpy.zeros(X_test.shape[0], bool)
  float_predictions = []
  for i, str_prediction in enumerate(str_predictions):
    try:
      float_prediction = float(str_prediction)
      float_predictions.append(float_prediction)
    except ValueError:
      illegal_predictions_mask[i] = True
  illegal_ratio = illegal_predictions_mask.sum() / X_test.shape[0]
  print(f"Percentage of illegal predictions: {illegal_ratio * 100}%")

  y_pred = numpy.array(float_predictions, numpy.float32)
  y = numpy.apply_along_axis(decode, 1, y_test).astype(numpy.float32)
  y = y[~illegal_predictions_mask]

  accuracy = (y == y_pred).sum() / X_test.shape[0]
  print(f"Accuracy: {accuracy:.2f}")

  y_denominator = numpy.copy(y)
  y_denominator[y == 0] = 1

  fractional_difference = (y_pred - y) / y_denominator

  print(f"Mean fractional error: {fractional_difference.mean()}")

  plt.hist(fractional_difference, bins=1000)
  plt.title("Global view of the distribution of fractional errors")
  plt.xlabel("Fractional error")
  plt.ylabel("Example count")
  plt.xscale("symlog")
  plt.yscale("log")
  plt.show()

  plt.hist(fractional_difference, bins=20, range=(-0.1, 0.1))
  plt.title("Zoomed view around 0 of the distribution of fractional errors")
  plt.xlabel("Fractional error")
  plt.ylabel("Example count")
  plt.show()


evaluate()